# تدريب Intent Classifier (Support/Sales)
SYNOVA — Part A

In [3]:
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

In [14]:
ID2LABEL = {0: "support", 1: "sales"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

print(ID2LABEL)

{0: 'support', 1: 'sales'}


In [21]:
df = pd.read_csv("data/intent_train.csv")
df["label"] = df["label"].map({"support": 0, "sales": 1})
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    89 non-null     object
 1   label   89 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.5+ KB


In [22]:
dataset = Dataset.from_pandas(df, preserve_index=False)

dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset


train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print(train_df.head())
print(test_df.head())
print("=="*50)
print(train_df['label'].value_counts())
print(test_df['label'].value_counts())
print("=="*50)
print(train_df.isnull().sum())
print(test_df.isnull().sum())
print("=="*50)
train_df = train_df.drop_duplicates()
print(train_df.duplicated().sum())
print("=="*100)
print(train_df["text"].str.len().describe())
print((train_df["text"].str.strip() == "").sum())

                                                text  label
0  can i return a product if it was damaged durin...      0
1         how can i apply for a job at your company?      0
2  can i request a product that is currently out ...      1
3  can i request a product that is listed as 'lim...      1
4  can i return a product if it was purchased wit...      0
                                                text  label
0  can i request a product that is listed as 'out...      1
1  can i return a product if i no longer have the...      0
2                          how can i track my order?      0
3  can i return a product if it was a final sale ...      0
4         how do i unsubscribe from your newsletter?      0
label
1    40
0    31
Name: count, dtype: int64
label
1    9
0    9
Name: count, dtype: int64
text     0
label    0
dtype: int64
text     0
label    0
dtype: int64
0
count    71.000000
mean     54.408451
std      18.503727
min      21.000000
25%      38.000000
50%      53.000000


In [23]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

tokenized_dataset = dataset.map(tokenize, batched=True)

c:\Users\user\OneDrive\Desktop\Capstone\main-repo\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

Map: 100%|██████████| 71/71 [00:00<00:00, 1915.82 examples/s]

Map: 100%|██████████| 18/18 [00:00<00:00, 2758.00 examples/s]


In [24]:
id2label = {0: "support", 1: "sales"}
label2id = {"support": 0, "sales": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")
    return {"accuracy": accuracy, "f1": f1}

In [26]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

c:\Users\user\OneDrive\Desktop\Capstone\main-repo\.venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
  0%|          | 0/18 [06:53<?, ?it/s]
                                      
  0%|          | 0/18 [01:38<?, ?it/s]        

{'loss': 0.7031, 'grad_norm': 2.7619516849517822, 'learning_rate': 3.611111111111111e-05, 'epoch': 0.56}





                                      

                                       
  0%|          | 0/18 [01:46<?, ?it/s]        



{'eval_loss': 0.6207802891731262, 'eval_accuracy': 0.5, 'eval_f1': 0.3333333333333333, 'eval_runtime': 1.0454, 'eval_samples_per_second': 17.219, 'eval_steps_per_second': 2.87, 'epoch': 1.0}


                                      
  0%|          | 0/18 [01:50<?, ?it/s]         

...




                                      

                                       
  0%|          | 0/18 [02:10<?, ?it/s]         



{'eval_loss': 0.5223878622055054, 'eval_accuracy': 0.7777777777777778, 'eval_f1': 0.775, 'eval_runtime': 1.0456, 'eval_samples_per_second': 17.215, 'eval_steps_per_second': 2.869, 'epoch': 2.0}


                                      
100%|██████████| 18/18 [00:46<00:00,  2.57s/it]

{'train_runtime': 46.3377, 'train_samples_per_second': 3.064, 'train_steps_per_second': 0.388, 'train_loss': 0.586826377444797, 'epoch': 2.0}


TrainOutput(global_step=18, training_loss=0.586826377444797, metrics={'train_runtime': 46.3377, 'train_samples_per_second': 3.064, 'train_steps_per_second': 0.388, 'train_loss': 0.586826377444797, 'epoch': 2.0})

In [27]:
results = trainer.evaluate()
print(results)

100%|██████████| 3/3 [00:00<00:00,  4.04it/s]

{'eval_loss': 0.5223878622055054, 'eval_accuracy': 0.7777777777777778, 'eval_f1': 0.775, 'eval_runtime': 1.3363, 'eval_samples_per_second': 13.47, 'eval_steps_per_second': 2.245, 'epoch': 2.0}


In [28]:
trainer.save_model("./intent_classifier_model")
tokenizer.save_pretrained("./intent_classifier_model")
print("تم الحفظ في مجلد intent_classifier_model")

تم الحفظ في مجلد intent_classifier_model


In [29]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./intent_classifier_model")

print(classifier("How much does this cost?"))
print(classifier("My order arrived broken"))
print(classifier("I want to buy a new product"))

[{'label': 'sales', 'score': 0.5223268866539001}]
[{'label': 'sales', 'score': 0.5030010342597961}]
[{'label': 'sales', 'score': 0.5949702262878418}]
